In [0]:
CATALOG = dbutils.widgets.get("catalog")
GOLD = F"{CATALOG}.skybound_gold"

In [0]:
# ── Schema-level grants ────────────────────────────────────────────────

grants = [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO senior_analysts",
    f"GRANT USE SCHEMA ON SCHEMA {GOLD} TO senior_analysts",
    f"GRANT SELECT ON TABLE {GOLD}.agg_current_airport_weather TO senior_analysts",
    f"GRANT SELECT ON TABLE {GOLD}.fact_airport_weather_reports TO senior_analysts",
    f"GRANT SELECT ON TABLE {GOLD}.agg_country_current TO senior_analysts",
    f"GRANT SELECT ON TABLE {GOLD}.dim_airport TO senior_analysts",
    f"GRANT SELECT ON TABLE {GOLD}.dim_flight_category TO senior_analysts",
]
for stmt in grants:
    spark.sql(stmt)

In [0]:
# ── Row filter function ─────────────────────────────────────────────

spark.sql(f"""
        CREATE OR REPLACE FUNCTION {GOLD}.row_filter_country(country_iso STRING)
        RETURNS BOOLEAN
        RETURN 
            CASE WHEN current_user() IN (
                'yuriy.verbitskyi@softserve.academy',
                'k.ilya.v@softserve.academy',
                'valeriialiebiedie179@softserve.academy'
                ) THEN True ELSE False END
            OR IS_ACCOUNT_GROUP_MEMBER('senior_analysts')
            OR IS_ACCOUNT_GROUP_MEMBER(CONCAT('country_', country_iso))
    """)

spark.sql(f"""
        ALTER TABLE {GOLD}.agg_current_airport_weather
        SET ROW FILTER {GOLD}.row_filter_country ON (airport_country)
    """)
spark.sql(f"""
        ALTER TABLE {GOLD}.fact_airport_weather_reports
        SET ROW FILTER {GOLD}.row_filter_country ON (station_id)
    """)

In [0]:
# ── Column masking ──────────────────────────────────────────────────

spark.sql(f"""
        CREATE OR REPLACE FUNCTION {GOLD}.mask_raw_metar(raw STRING)
        RETURNS STRING
        RETURN CASE
            WHEN current_user() IN (
                'yuriy.verbitskyi@softserve.academy',
                'k.ilya.v@softserve.academy'
                ) THEN raw
            WHEN current_user() = 'valeriialiebiedie179@softserve.academy'
                THEN CONCAT(LEFT(raw, 3), '***')
            ELSE '*** RESTRICTED ***'
        END
    """)

spark.sql(f"""
        ALTER TABLE {GOLD}.fact_airport_weather_reports
        ALTER COLUMN raw_ob SET MASK {GOLD}.mask_raw_metar
    """)